## 0 · Setup prostředí

Workbench běží na RHOAI _Standard Data Science Notebook_ image. Pandas,
numpy, scikit-learn, matplotlib, boto3 a od RHOAI 3.2 i **MLflow klient**
(Red Hat build, viz [KCS 7136121](https://access.redhat.com/articles/7136121))
jsou už předinstalované. Žádný `pip install` se v notebooku nedělá.

Platforma navíc workbench podu sama nastavuje `MLFLOW_TRACKING_URI`,
`MLFLOW_TRACKING_AUTH=kubernetes-namespaced` a namountuje SA token —
tracking server je tím pádem k mání bez jediného řádku konfigurace.

In [ ]:
# Ověř, že workbench je naladěn proti platform-managed MLflow.
# RHOAI 3.4 přidá tyhle env vars automaticky, když je `mlflowoperator`
# zapnutý v DataScienceCluster (kontrola: `oc get dsc -A`).
import os, mlflow
print(f'mlflow client      : {mlflow.__version__}')
print(f'MLFLOW_TRACKING_URI: {os.environ.get("MLFLOW_TRACKING_URI", "(unset)")}')
print(f'MLFLOW_TRACKING_AUTH: {os.environ.get("MLFLOW_TRACKING_AUTH", "(unset)")}')

# 02 · Trénink modelu pro detekci anomálií

**Scénář ze slide 20:** _reprodukovatelnost a sdílení_ · _auditovatelnost a compliance_

Z průzkumového notebooku víme, na co se dívat. Teď natrénujeme model,
který kombinuje **všechny** signály najednou a každé faktuře přidělí
skóre.

Vše, co teď spustíme, se zaloguje do **MLflow** — datasety, parametry,
metriky, artefakty. To je naše evidence pro audit (NIS2 / AI Act):
kdo, kdy, na čem natrénoval, s jakým výsledkem.

## 2.1 Načtení dat a feature engineering

Featury držíme v `src/invoice_features.py`, aby se trénink i inference
shodly. Žádné copy-paste mezi notebooky.

In [ ]:
import os, sys
sys.path.insert(0, '../src')

from invoice_features import load_invoices, build_features, FEATURE_COLUMNS

if os.environ.get('AWS_S3_BUCKET'):
    from invoice_features import get_s3_client, get_bucket
    s3 = get_s3_client()
    df = load_invoices('invoices.csv', s3_client=s3, bucket=get_bucket())
else:
    df = load_invoices('../data/invoices.csv')

print(f'Trénovací data: {len(df):,} faktur')

In [ ]:
X, meta = build_features(df)
print('Feature columns:', FEATURE_COLUMNS)
print('Shape:', X.shape)

## 2.2 Publikace features do Feast (registratura + online store)

Featury jsou hotové. Teď je zveřejníme přes **Feast feature store** —
platform komponenta RHOAI 3.4 (`feastoperator: Managed` v DSC).
Tím získáme:

- **Single source of truth** pro definice featur — `src/feast_repo/`
  v gitu, operator je dotáhne při startu (FeatureStore CR ji klonuje).
- **Auditovatelný registr** — `feast list feature-views` ti řekne,
  který tým má kterou featuru, od kdy, s jakým TTL.
- **Online store** s low-latency lookupem pro produkční inference
  (KServe rollout v beat 6 → MLServer si v reálu sahá pro featury
  právě sem, ne do CSV).

Operator už registry zapsal z gitu (server-side). Z workbench dělá
vlastní `feast apply` proti `src/feast_repo/` lokálně, ať notebook
běží end-to-end i bez čekání na clusterový redeploy.

In [ ]:
from feast_io import (write_feature_parquet, apply_feature_definitions,
                      materialize_incremental, open_local_store)

# 1) Napiš feature parquet — to je offline source, který FileSource v
#    feast_repo/data_sources.py referencuje. Cesta v parquet odpovídá
#    s3://$AWS_S3_BUCKET/feast/invoice_features.parquet.
if os.environ.get('AWS_S3_BUCKET'):
    s3 = get_s3_client()
    feat_df, _ = write_feature_parquet(df, s3_client=s3, bucket=get_bucket())
else:
    feat_df, _ = write_feature_parquet(df,
                                       local_path='/tmp/feast/invoice_features.parquet')

print(f'Feature parquet: {len(feat_df):,} řádků × {feat_df.shape[1]} sloupců')
feat_df.head(3)

In [ ]:
# 2) feast apply — z .py souborů v feast_repo/ aktualizuje registry.
#    Idempotentní; běh trvá vteřiny.
apply_feature_definitions()

In [ ]:
# 3) feast materialize-incremental — z offline parquet do SQLite online
#    store. Feast si pamatuje materialize watermark per FV, takže další
#    běh nahraje jen to nové.
materialize_incremental()

In [ ]:
# 4) Sanity check: online lookup. Workbench drží svůj lokální Feast
#    klient (registry + online store na PVC), který umí get_online_features.
#    Pro produkční cestu (MLServer) by se použila operator-managed
#    online endpoint — viz deploy/09-feature-store.yaml.
fs = open_local_store()
sample_id = df['invoice_id'].sample(1, random_state=0).iloc[0]

resp = fs.get_online_features(
    features=[f'invoice_features:{c}' for c in FEATURE_COLUMNS],
    entity_rows=[{'invoice_id': sample_id}],
).to_dict()

print(f'Online lookup pro invoice_id={sample_id}:')
for k, v in resp.items():
    print(f'  {k:<32} {v}')

## 2.3 Trénink Isolation Forest

Pro tabulární anomálie je _Isolation Forest_ rozumný start —
rychlý, nepotřebuje labely, dobře interpretovatelný.
Parametry necháme **otevřené v buňce**, ať si je business analytik
může upravit bez čtení dokumentace.

In [ ]:
# ---- PARAMETRY (uprav, pokud chceš jiný profil) ------------------
CONTAMINATION = 0.03    # očekávaný podíl anomálií (0–0.1)
N_ESTIMATORS  = 200     # počet stromů — víc = stabilnější, pomalejší
MAX_SAMPLES   = 'auto'  # 'auto' = min(256, n_samples)
RANDOM_STATE  = 42
# ------------------------------------------------------------------

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('iforest', IsolationForest(
        contamination=CONTAMINATION,
        n_estimators=N_ESTIMATORS,
        max_samples=MAX_SAMPLES,
        random_state=RANDOM_STATE,
    )),
])
pipe.fit(X)
print('Trénink hotov.')

## 2.4 Vyhodnocení proti známým anomáliím

V demo datech labely _máme_ — využijeme je k report-cardu. V produkci
by se sem dosadila zpětná vazba schvalovatelů (controller reviewer flag).

In [ ]:
import numpy as np
from sklearn.metrics import (precision_recall_curve, roc_auc_score,
                              average_precision_score)

scores = -pipe.decision_function(X)   # vyšší = podezřelejší
y_true = df['is_anomaly'].values

roc_auc = roc_auc_score(y_true, scores)
pr_auc  = average_precision_score(y_true, scores)
print(f'ROC-AUC: {roc_auc:.3f}')
print(f'PR-AUC : {pr_auc:.3f}')

In [ ]:
# Precision a recall pro různé velikosti review queue
# (= kolik podezřelých faktur si controller prokliká).
#   precision = z těch, co jsme flagli, kolik bylo opravdu špatně
#   recall    = z všech skutečných anomálií kolik jsme zachytili
order  = np.argsort(scores)[::-1]
n_anom = int(y_true.sum())
print(f'{"Top k":>6} | {"precision":>9} | {"recall":>7} | TP / k   (z {n_anom} skutečných)')
print('-' * 64)
for k in (50, 100, 200, 500):
    tp        = int(y_true[order[:k]].sum())
    precision = tp / k
    recall    = tp / n_anom
    print(f'{k:>6} | {precision:>9.2%} | {recall:>7.2%} | {tp:>3} / {k}')

## 2.5 Logování do MLflow

Tohle je krok, který dělá z notebooku **auditovatelný artefakt**.
MLflow běží jako platformní komponenta RHOAI (managed přes
`mlflowoperator` v DataScienceCluster). UI je dostupné z RHOAI dashboardu
(`Applications → MLflow`), API přes Service `mlflow.redhat-ods-applications.svc:8443`.

Klient se autentizuje SA tokenem workbenche; server validuje práva přes
`SelfSubjectAccessReview`, a **namespace = workspace** — všechny runs
z tohoto notebooku skončí izolované v projektu `rh-ai-workshop`.

In [ ]:
try:
    import mlflow
    import mlflow.sklearn

    # MLFLOW_TRACKING_URI / _AUTH si nastavuje platforma; lokální fallback
    # je `file:./mlruns`, ať notebook běží i mimo cluster.
    if not os.environ.get('MLFLOW_TRACKING_URI'):
        mlflow.set_tracking_uri('file:./mlruns')
    mlflow.set_experiment('invoice-anomaly')

    with mlflow.start_run(run_name='isolation-forest-baseline') as run:
        mlflow.log_params({
            'contamination': CONTAMINATION,
            'n_estimators': N_ESTIMATORS,
            'max_samples': str(MAX_SAMPLES),
            'features': ','.join(FEATURE_COLUMNS),
            'n_train_rows': len(df),
        })
        mlflow.log_metrics({
            'roc_auc': roc_auc,
            'pr_auc': pr_auc,
            'n_anomalies_true': int(y_true.sum()),
        })
        mlflow.sklearn.log_model(pipe, artifact_path='model')
        run_id = run.info.run_id
        print(f'MLflow run logged: {run_id}')
        print(f'Tracking URI:     {mlflow.get_tracking_uri()}')
except Exception as exc:
    print('MLflow nedostupný v této session — pokračujeme bez něj:', exc)
    run_id = None

## 2.6 Uložení modelu (a meta tabulek) do S3

Vedle MLflow registru ukládáme i přímo do bucketu — pro pipeline,
která spouští skórování v nightly batchi, je to jednodušší než tahat
binárku z MLflow.

In [ ]:
import joblib, json, io

artifact = {
    'pipeline': pipe,
    'feature_columns': FEATURE_COLUMNS,
    'category_index': meta['category_index'],
    'vendor_freq_table': meta['vendor_freq_table'].to_dict(),
    'category_stats': meta['category_stats'].to_dict(orient='index'),
    'metrics': {'roc_auc': float(roc_auc), 'pr_auc': float(pr_auc)},
    'mlflow_run_id': run_id,
}

out_path = '../data/invoice_anomaly_model.joblib'
joblib.dump(artifact, out_path)
print('Model uložen lokálně do:', out_path)

if os.environ.get('AWS_S3_BUCKET'):
    s3 = get_s3_client()
    buf = io.BytesIO(); joblib.dump(artifact, buf); buf.seek(0)
    s3.put_object(Bucket=get_bucket(), Key='models/invoice_anomaly_model.joblib',
                  Body=buf.getvalue())
    print('Model uložen i do S3 bucketu')

## 2.7 Co máme z pohledu compliance

- Notebook (kód) → Git.
- Definice featur → Git **+** Feast registry (operator-managed).
- Feature values (snapshot) → S3 parquet **+** Feast online store.
- Parametry + metriky + dataset signature → MLflow.
- Binárka modelu → S3 (versionovaný klíč).
- V dalším kroku model zaregistrujeme do **Model Registry** se stage
  `Staging` → po review business sponzora `Production`.

**To je celá audit trail jedním notebookem.**